In [ ]:
import pandas as pd
import re

# ============================================================
# Configuration
# ============================================================
INPUT_FILE = 'npidata_pfile_20050523-20260308.csv'
OUTPUT_FILE = 'radiologist_pathologist_filtered_2026_03.csv'
CHUNKSIZE = 100_000

TARGET_CODES = {
    '2085R0202X',  # Diagnostic Radiology Physician - PRIMARY ANALYSIS
    '207ZP0102X',  # Anatomic Pathology & Clinical Pathology - robustness check
    '207ZP0101X',  # Anatomic Pathology - robustness check
    '207ZC0500X',  # Cytopathology - robustness check
}

# 50 states + DC only (excludes PR, GU, VI, AS, MP, and other territories)
US_STATES = {
    'AL','AK','AZ','AR','CA','CO','CT','DE','DC','FL','GA','HI','ID',
    'IL','IN','IA','KS','KY','LA','ME','MD','MA','MI','MN','MS','MO',
    'MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA',
    'RI','SC','SD','TN','TX','UT','VT','VA','WA','WV','WI','WY'
}

TAXONOMY_COLS = [f'Healthcare Provider Taxonomy Code_{i}' for i in range(1, 16)]
TAXONOMY_SWITCH_COLS = [f'Healthcare Provider Primary Taxonomy Switch_{i}' for i in range(1, 16)]

KEEP_COLS = [
    'NPI',
    'Replacement NPI',
    'Entity Type Code',
    'Is Sole Proprietor',
    'Provider Organization Name (Legal Business Name)',
    'Provider Last Name (Legal Name)',
    'Provider First Name',
    'Provider Name Suffix Text',
    'Provider Credential Text',
    'Provider Sex Code',
    'Provider Enumeration Date',
    'Provider First Line Business Practice Location Address',
    'Provider Second Line Business Practice Location Address',
    'Provider Business Practice Location Address City Name',
    'Provider Business Practice Location Address State Name',
    'Provider Business Practice Location Address Postal Code',
    'Provider Business Practice Location Address Country Code (If outside U.S.)',
    'NPI Deactivation Date',
    'NPI Reactivation Date',
] + TAXONOMY_COLS + TAXONOMY_SWITCH_COLS

# ============================================================
# Main Processing
# Note: No credential-based filtering is applied.
# Provider specialty is identified solely by NPPES taxonomy code
# (2085R0202X = Diagnostic Radiology Physician).
# As NPPES data are self-reported, a small degree of
# misclassification cannot be excluded; this is noted as a
# limitation in the Methods section.
# ============================================================
results = []
total_read = 0
total_matched = 0

print("Starting processing...")

for chunk in pd.read_csv(INPUT_FILE,
                          usecols=KEEP_COLS,
                          chunksize=CHUNKSIZE,
                          low_memory=False,
                          dtype=str):

    total_read += len(chunk)

    # Step 1: Individual providers only (Entity Type Code = 1)
    chunk = chunk[chunk['Entity Type Code'] == '1']

    # Step 2: Active providers only (no deactivation date)
    chunk = chunk[chunk['NPI Deactivation Date'].isna()]

    # Step 3: 50 states + DC only
    state_col = 'Provider Business Practice Location Address State Name'
    chunk = chunk[chunk[state_col].isin(US_STATES)]

    # Step 4: Any of 15 taxonomy columns matches a target code
    mask = chunk[TAXONOMY_COLS].isin(TARGET_CODES).any(axis=1)
    matched = chunk[mask].copy()

    if len(matched) > 0:
        results.append(matched)
        total_matched += len(matched)

    print(f"  Rows read: {total_read:,} | Rows matched: {total_matched:,}", end='\r')

print(f"\nProcessing complete. Total read: {total_read:,} | Total matched: {total_matched:,}")

# ============================================================
# Combine and Clean Up
# ============================================================
df_out = pd.concat(results, ignore_index=True)

# Extract 5-digit ZIP code
df_out['zip5'] = df_out['Provider Business Practice Location Address Postal Code'].str[:5]

# Identify matched taxonomy code
def get_matched_taxonomy(row):
    for col in TAXONOMY_COLS:
        if row[col] in TARGET_CODES:
            return row[col]
    return None

df_out['matched_taxonomy'] = df_out.apply(get_matched_taxonomy, axis=1)

# Flag whether matched taxonomy is primary specialty
def get_primary_switch(row):
    for i, col in enumerate(TAXONOMY_COLS):
        if row[col] in TARGET_CODES:
            return row[TAXONOMY_SWITCH_COLS[i]]
    return None

df_out['matched_taxonomy_is_primary'] = df_out.apply(get_primary_switch, axis=1)

# ============================================================
# Summary
# ============================================================
print("\nRecord count by taxonomy code:")
print(df_out['matched_taxonomy'].value_counts())

print("\nIs target taxonomy primary specialty?")
print(df_out['matched_taxonomy_is_primary'].value_counts())

print("\nCredential distribution - 2085R0202X (top 20):")
rad_only = df_out[df_out['matched_taxonomy'] == '2085R0202X']
print(rad_only['Provider Credential Text'].value_counts().head(20))

print("\nSex distribution:")
print(df_out['Provider Sex Code'].value_counts())

print("\nState distribution (top 10):")
print(df_out['Provider Business Practice Location Address State Name'].value_counts().head(10))

# ============================================================
# Save Output
# ============================================================
df_out.to_csv(OUTPUT_FILE, index=False)
print(f"\nSaved to: {OUTPUT_FILE}")
print(f"Total records saved: {len(df_out):,}")
